# EP3 — gestão de estoque e relatórios

Exercício de programação para ler arquivos CSV de lojas, clientes, produtos, inventário e operações; atualizar estoque/caixa; e produzir relatórios de movimentação.

**Arquivos de entrada necessários:** `inventario.csv`, `operacoes.csv`, `clientes.csv`, `lojas.csv` e `produtos.csv`. Esses arquivos não estavam no ZIP recebido e não foram inventados nem incluídos. Coloque-os na mesma pasta do notebook antes de executar.

Para facilitar a leitura, esta versão mantém a implementação mais recente do notebook recebido e remove células que continham versões anteriores repetidas e um teste de entrada independente.

In [ ]:
import numpy as np
import os

def le_dados():
    inventario = np.genfromtxt("inventario.csv", dtype=np.int32, skip_header=1, delimiter=",")
    operacoes = np.genfromtxt("operacoes.csv", dtype=np.int32, skip_header=1, delimiter=",")
    clientes = np.genfromtxt("clientes.csv", dtype='U40', skip_header=1, delimiter=",")
    lojas = np.genfromtxt("lojas.csv", dtype='U40', skip_header=1, delimiter=",")
    produtos = np.genfromtxt("produtos.csv", dtype='U40', skip_header=1, delimiter=",")
    return inventario, operacoes, clientes, lojas, produtos

def preco_item(inventario, id_loja, id_produto):
    test = (inventario[:, 0] == id_loja) & (inventario[:, 1] == id_produto)
    linhas_correspondentes = inventario[test]
    if linhas_correspondentes.shape[0] > 0:
        return linhas_correspondentes[0, 2]
    else:
        return None

def nome_loja(lojas, id_loja):
    test = (lojas[:, 0] == str(id_loja))
    linhas_correspondentes = lojas[test]
    if linhas_correspondentes.shape[0] > 0:
        return linhas_correspondentes[0, 1]
    else:
        return None

def nome_produto(produtos, id_produto):
    test = (produtos[:, 0] == str(id_produto))
    linhas_correspondentes = produtos[test]
    if linhas_correspondentes.shape[0] > 0:
          return linhas_correspondentes[0, 1]
    else:
          return None

def nome_cliente(clientes, id_cliente):
    test = (clientes[:, 0] == str(id_cliente))
    linhas_correspondentes = clientes[test]
    if linhas_correspondentes.shape[0] > 0:
          return linhas_correspondentes[0, 1]
    else:
          return None

def remove_estoque(inventario, id_loja, id_produto, quantidade):
    test = (inventario[:, 0] == id_loja) & (inventario[:, 1] == id_produto)
    if np.any(test):
        indice = np.where(test)[0][0]
        if inventario[indice, 3] >= quantidade:
            inventario[indice, 3] -= quantidade
            return True
        else:
            return False
    return False

def soma_caixa(lojas, id_loja, valor):
    teste = (lojas[:, 0] == str(id_loja))
    if np.any(teste):
        indice = np.where(teste)[0][0]
        lojas[indice, 2] = str(int(lojas[indice, 2]) + valor)
        return True
    return False

def atualiza_estoques(inventario, operacoes):
    for operacao in operacoes:
        id_loja = operacao[1]
        id_produto = operacao[2]
        quantidade = operacao[3]

        teste = (inventario[:, 0] == id_loja) & (inventario[:, 1] == id_produto)
        indices = np.where(teste)[0]

        if indices.size > 0:
            indice = indices[0]
            estoque_atual = inventario[indice, 3]

            if estoque_atual >= quantidade:
                inventario[indice, 3] -= quantidade
            else:
                print(f"Estoque insuficiente para o produto {id_produto} na loja {id_loja}.")
        else:
            print(f"O produto {id_produto} não foi encontrado na loja {id_loja}.")
            continue

    np.savetxt("novo_inventario.csv", inventario, fmt="%s", delimiter=",",
               header="id_loja,id_produto,categoria,quantidade", comments="")


def atualiza_caixas(inventario, lojas, operacoes):
    for operacao in operacoes:
        id_produto = operacao[2]
        id_loja = operacao[1]
        quantidade = operacao[3]
        preco_produto = preco_item(inventario, id_loja, id_produto)

        if preco_produto is None:
            print(f"O produto {id_produto} não foi encontrado na loja {id_loja}.")
            continue

        valor_operacao = preco_produto * quantidade
        indice_loja = np.where(lojas[:, 0].astype(int) == id_loja)[0]

        if indice_loja.size > 0:
            indice = indice_loja[0]
            caixa_atual = int(lojas[indice, 2])
            lojas[indice, 2] = str(caixa_atual + valor_operacao)
        else:
            print(f"Loja {id_loja} não encontrada.")

    if not os.path.exists("novo_lojas.csv"):
        print("Arquivo 'novo_lojas.csv' não encontrado, criando novo arquivo.")
    np.savetxt("novo_lojas.csv", lojas, fmt="%s", delimiter=",", header="id_loja,nome_loja,caixa", comments="")
    print("Arquivo 'novo_lojas.csv' atualizado com sucesso.")
def vendas_por_produto(operacoes, produtos):
    nprodutos = produtos.shape[0]
    vendas = np.zeros([nprodutos, 2], dtype=np.int32)
    for i in range(nprodutos):
        vendas[i,0] = produtos[i,0]
    nops = operacoes.shape[0]
    for i in range(nops):
        id_cliente, id_loja, id_produto, quantidade = operacoes[i]
        for j in range(nprodutos):
            if vendas[j,0] == id_produto:
                vendas[j,1] += quantidade
    return vendas

def quantidade_vendidos(vendas, id_produto):
    for i in range(vendas.shape[0]):
        if vendas[i, 0] == id_produto:
            return vendas[i, 1]
    return 0

def reposicao(inventario, operacoes, produtos):
    vendas = vendas_por_produto(operacoes, produtos)
    reposicao_estoque = []
    for i in range(inventario.shape[0]):
        id_loja = inventario[i, 0]
        id_produto = inventario[i, 1]
        estoque = inventario[i, 3]
        if estoque == 0:
            quantidade_vendida = quantidade_vendidos(vendas, id_produto)
            quantidade_reposicao = (quantidade_vendida // 2) + 1
            reposicao_estoque.append([id_loja, id_produto, quantidade_reposicao])
        else:
            reposicao_estoque.append([id_loja, id_produto, 0])
    reposicao_estoque = np.array(reposicao_estoque, dtype=np.int32)
    np.savetxt("reposicao_estoque.csv", reposicao_estoque, fmt="%d", delimiter=",", header="id_loja,id_produto,quantidade", comments="")
    return reposicao_estoque

def faturamento_por_loja(inventario, operacoes, lojas):
    faturamento = {int(loja[0]): 0 for loja in lojas}
    for operacao in operacoes:
        id_cliente, id_loja, id_produto, quantidade = operacao
        preco = preco_item(inventario, id_loja, id_produto)
        faturamento_op = preco * quantidade
        if id_loja in faturamento:
            faturamento[id_loja] += faturamento_op

    lojas_faturamento = [[id_loja, faturamento[id_loja]] for id_loja in faturamento]
    lojas_faturamento = np.array(lojas_faturamento, dtype=np.int32)
    return lojas_faturamento

def relatorio_lojas(inventario, operacoes, lojas):
    faturamentos = faturamento_por_loja(inventario, operacoes, lojas)
    xlojas = lojas.shape[0]
    for i in range(xlojas):
        id_loja = faturamentos[i,0]
        valor = faturamentos[i,1]
        loja = nome_loja(lojas, id_loja)
        if valor > 0:
            print(f'A loja "{loja}" faturou R${valor}')
        else:
            print(f'A loja "{loja}" não teve vendas')

def gasto_por_cliente(inventario, operacoes, clientes):
    gastos = {}
    for operacao in operacoes:
        id_cliente, id_loja, id_produto, quantidade = operacao
        preco = preco_item(inventario, id_loja, id_produto)
        gasto_op = preco * quantidade
        if id_cliente in gastos:
            gastos[id_cliente] += gasto_op
        else:
            gastos[id_cliente] = gasto_op

    clientes_gastos = []
    for cliente in clientes:
        id_cliente = int(cliente[0])
        gasto = gastos.get(id_cliente, 0)
        clientes_gastos.append([id_cliente, gasto])

    clientes_gastos = np.array(clientes_gastos, dtype=np.int32)
    return clientes_gastos

def relatorios_individuais(inventario, operacoes, clientes, lojas):
    gastos = gasto_por_cliente(inventario, operacoes, clientes)
    xclientes = gastos.shape[0]
    clientes_ativos = []
    faturamento = 0
    for i in range(xclientes):
        if gastos[i,1] > 0:
            id_cliente = gastos[i,0]
            gasto = gastos[i,1]
            cliente = nome_cliente(clientes, id_cliente)
            faturamento += gasto
            clientes_ativos.append(cliente)
            print(f"{cliente} gastou no total R${gasto}")
    return faturamento, clientes_ativos

def relatorio_clientes(inventario, operacoes, clientes, lojas):
    faturamento, clientes_ativos = relatorios_individuais(inventario,operacoes,clientes, lojas)

    if len(clientes_ativos) == 0:
        print("Nenhum cliente fez compras no shopping hoje!")
        return

    nomes = ', '.join(clientes_ativos)
    ult_virgula = nomes.rfind(", ")
    nomes = (nomes[:ult_virgula] + " e " + nomes[ult_virgula +2:])
    print(f"{len(clientes_ativos)} clientes fizeram compras no shopping:",nomes)
    media = faturamento / len(clientes_ativos)
    print(f"Os clientes gastaram em média R${media:.2f}")
    print(f"O faturamento total do shopping foi R${faturamento}")

def relatorio_estoques(reposicoes, lojas, produtos):
    xitems = reposicoes.shape[0]
    for i in range(xitems):
        id_loja, id_produto, quantidade = reposicoes[i]
        if quantidade > 0:
            loja = nome_loja(lojas, id_loja)
            produto = nome_produto(produtos, id_produto)
            print(f'A loja "{loja}" deve comprar {quantidade} '
                  f'unidades do produto {produto}')

def main():
    inventario, operacoes, clientes, lojas, produtos = le_dados()
    atualiza_estoques(inventario, operacoes)
    atualiza_caixas(inventario, lojas, operacoes)
    reposicoes = reposicao(inventario, operacoes, produtos)
    print("==== Relatório do dia ====")
    print()
    relatorio_clientes(inventario, operacoes, clientes, lojas)
    relatorio_lojas(inventario, operacoes, lojas)
    relatorio_estoques(reposicoes, lojas, produtos)

if __name__ == "__main__":
    main()